<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V2_AD_Only_Manneim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Zugangsberechtigung auf Drive

from google.colab import drive
drive.mount('/content/drive')

# Gezippte Daten werden entpackt und in hohes Verzeichnis gelegt

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"

!unzip -q "/content/data.zip" -d "/content"

!rm "/content/data.zip"
!rm "/content/_MACOSX"


Mounted at /content/drive
rm: cannot remove '/content/_MACOSX': No such file or directory


In [ ]:
# ## 0 — Setup & Imports

# %%
import os, glob, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from shapely import wkb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve, auc, f1_score, roc_auc_score,
    classification_report, roc_curve
)
import matplotlib.pyplot as plt

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device: cuda
  GPU: NVIDIA L4
  VRAM: 23.7 GB


In [ ]:
# ## 1 — Pfade & Konfiguration

# %%
# ── Pfade ──
DEMAND_BASE        = '/content/data/demand'
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'
GEO_INFO_PATH      = '/content/data/geo_information/geo_information.parquet'
WEATHER_PATH       = '/content/data/weather/weather.parquet'
HOLIDAYS_PATH      = '/content/data/holidays/holidays.parquet'
VACATIONS_PATH     = '/content/data/vacations/vacations.parquet'

OUTPUT_PATH        = '/content/data/Second_AD_Mannheim_only.parquet'

# ── Konfiguration ──
CITY               = 'Mannheim'
NETWORK_NAME       = 'Mannheim'          # Wie es in der network_name-Spalte steht
WEATHER_STATION_ID = 292348               # Wetterstation location_id für Mannheim
FEDERAL_STATE      = 'Baden-Württemberg'  # Für Feiertage/Ferien-Filter

# AD-Hyperparameter
WINDOW_SIZE        = 24                   # Stunden (1 Tag)
LATENT_DIM         = 32
Z_TRAIN_THRESHOLD  = 2.0                  # |z| <= 2 → normal (Training)
Z_ANOMALY_THRESHOLD = 3.0                 # |z| > 3 → anomal (Evaluation)
MIN_EVENTS_PER_DAY = 1.0                  # Mindestaktivität pro Station

# Training
BATCH_SIZE         = 8192
EPOCHS             = 30
LEARNING_RATE      = 1e-3
TRAIN_RATIO        = 0.67
VAL_RATIO          = 0.83                 # Rest = Test



In [ ]:
# ## 2 — Hilfsdaten laden (Stationen, Wetter, Feiertage)

# %%
# ── Station Names & Typ-Klassifikation ──
def classify_station(name: str) -> str:
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names['station_type'] = station_names['station_name'].apply(classify_station)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()
print(f'Station-Types: {station_names["station_type"].value_counts().to_dict()}')

# %%
# ── Wetter laden & auf Stunde aggregieren ──
weather = pd.read_parquet(WEATHER_PATH)
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)

# Nur Mannheim-Wetterstation
weather_ma = weather[weather['location_id'] == WEATHER_STATION_ID].copy()
weather_ma['hour_ts'] = weather_ma['timestamp'].dt.floor('h')

weather_hourly = (
    weather_ma
    .groupby('hour_ts')
    .agg(
        temperature   = ('temperature', 'mean'),
        humidity      = ('humidity', 'mean'),
        precipitation = ('precipitation', 'sum'),
        wind_speed    = ('wind_speed', 'mean'),
    )
    .reset_index()
)
print(f'Wetter stündlich: {len(weather_hourly):,} Zeilen | '
      f'{weather_hourly["hour_ts"].min().date()} – {weather_hourly["hour_ts"].max().date()}')

# %%
# ── Feiertage & Ferien (nur BaWü) ──
holidays  = pd.read_parquet(HOLIDAYS_PATH)
vacations = pd.read_parquet(VACATIONS_PATH)
holidays['start_date']  = pd.to_datetime(holidays['start_date'])
holidays['end_date']    = pd.to_datetime(holidays['end_date'])
vacations['start_date'] = pd.to_datetime(vacations['start_date'])
vacations['end_date']   = pd.to_datetime(vacations['end_date'])

holidays_bw  = holidays[holidays['federal_state_name'] == FEDERAL_STATE]
vacations_bw = vacations[vacations['federal_state_name'] == FEDERAL_STATE]

# Sets aller Feiertags-/Ferientage für schnellen Lookup
holiday_dates = set()
for _, row in holidays_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        holiday_dates.add(d.date())

vacation_dates = set()
for _, row in vacations_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        vacation_dates.add(d.date())

print(f'Feiertage BaWü: {len(holiday_dates)} Tage | Ferien BaWü: {len(vacation_dates)} Tage')


Station-Types: {'bike': 44747, 'real': 13040, 'virtual': 4827, 'recording': 1972, 'only_nums': 51}
Wetter stündlich: 24,913 Zeilen | 2023-04-01 – 2026-02-02
Feiertage BaWü: 167 Tage | Ferien BaWü: 277 Tage


In [ ]:
# ## 3 — Demand laden & auf Stunde aggregieren

# %%
# ── Mannheim-Demand laden ──
files = glob.glob(os.path.join(DEMAND_BASE, CITY, '**', '*.parquet'), recursive=True)
if not files:
    files = glob.glob(os.path.join(DEMAND_BASE, CITY, '*.parquet'))
print(f'Parquet-Dateien gefunden: {len(files)}')

cols = ['network_name', 'timestamp', 'station_id', 'station_name_id',
        'location_id', 'n_lends', 'n_returns']
demand_raw = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
demand_raw['timestamp'] = pd.to_datetime(demand_raw['timestamp'], utc=True)

# Station-Typ zuweisen & filtern
demand_raw['station_type'] = demand_raw['station_name_id'].map(type_lookup).fillna('unknown')
demand = demand_raw[demand_raw['station_type'] == 'real'].copy()

print(f'Demand roh:      {len(demand_raw):,} Zeilen')
print(f'Demand (real):   {len(demand):,} Zeilen')
print(f'Stationen:       {demand["station_id"].nunique()}')
print(f'Zeitraum:        {demand["timestamp"].min().date()} – {demand["timestamp"].max().date()}')

# %%
# ── Stündliche Aggregation pro Station ──
demand['hour_ts'] = demand['timestamp'].dt.floor('h')

hourly = (
    demand
    .groupby(['station_id', 'station_name_id', 'location_id', 'hour_ts'])
    .agg(
        n_lends   = ('n_lends', 'sum'),
        n_returns = ('n_returns', 'sum'),
    )
    .reset_index()
)
hourly['total_demand'] = hourly['n_lends'] + hourly['n_returns']

print(f'Stündlich aggregiert: {len(hourly):,} Zeilen')

# ── Lücken füllen: Jede Station braucht JEDE Stunde ──
all_hours = pd.date_range(
    hourly['hour_ts'].min(),
    hourly['hour_ts'].max(),
    freq='h',
    tz='UTC'
)

# Erst auf station_id × hour_ts eindeutig aggregieren
# (falls ein station_id mehrere location_ids hat)
hourly_agg = (
    hourly
    .groupby(['station_id', 'hour_ts'])
    .agg(
        n_lends       = ('n_lends', 'sum'),
        n_returns     = ('n_returns', 'sum'),
        total_demand  = ('total_demand', 'sum'),
        station_name_id = ('station_name_id', 'first'),
        location_id     = ('location_id', 'first'),
    )
    .reset_index()
)
print(f'Nach Deduplizierung: {len(hourly_agg):,} Zeilen '
      f'(vorher: {len(hourly):,}, Duplikate entfernt: {len(hourly) - len(hourly_agg):,})')

# Station-Info für späteren Re-Join
station_info = (
    hourly_agg
    .groupby('station_id')
    .agg(station_name_id=('station_name_id', 'first'),
         location_id=('location_id', 'first'))
    .reset_index()
)

# Reindex mit sauberem eindeutigen Index
full_index = pd.MultiIndex.from_product(
    [station_info['station_id'].values, all_hours],
    names=['station_id', 'hour_ts']
)
hourly_full = (
    hourly_agg[['station_id', 'hour_ts', 'n_lends', 'n_returns', 'total_demand']]
    .set_index(['station_id', 'hour_ts'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Station-Info wieder dranhängen
hourly_full = hourly_full.merge(station_info, on='station_id', how='left')

print(f'Nach Lückenfüllung: {len(hourly_full):,} Zeilen')
print(f'Stunden pro Station: {len(all_hours):,}')
print(f'Stationen: {hourly_full["station_id"].nunique()}')
# %%
# ── Stationen mit zu wenig Aktivität entfernen ──
n_days = (all_hours[-1] - all_hours[0]).days + 1
station_activity = hourly_full.groupby('station_id')['total_demand'].sum()
min_events = int(n_days * MIN_EVENTS_PER_DAY)
active_stations = station_activity[station_activity >= min_events].index

hourly_full = hourly_full[hourly_full['station_id'].isin(active_stations)].copy()
print(f'Stationen nach Aktivitätsfilter (≥{MIN_EVENTS_PER_DAY}/Tag): '
      f'{hourly_full["station_id"].nunique()} '
      f'(entfernt: {len(station_activity) - len(active_stations)})')


Parquet-Dateien gefunden: 36
Demand roh:      2,683,584 Zeilen
Demand (real):   2,579,329 Zeilen
Stationen:       128
Zeitraum:        2023-03-31 – 2026-02-02
Stündlich aggregiert: 1,056,222 Zeilen
Nach Deduplizierung: 1,056,218 Zeilen (vorher: 1,056,222, Duplikate entfernt: 4)
Nach Lückenfüllung: 3,191,936 Zeilen
Stunden pro Station: 24,937
Stationen: 128
Stationen nach Aktivitätsfilter (≥1.0/Tag): 104 (entfernt: 24)


In [ ]:
# ## 4 — Feature Engineering

# %%
# ── Kalender-Features ──
hourly_full['hour_of_day'] = hourly_full['hour_ts'].dt.hour
hourly_full['day_of_week'] = hourly_full['hour_ts'].dt.dayofweek
hourly_full['month']       = hourly_full['hour_ts'].dt.month
hourly_full['is_weekend']  = (hourly_full['day_of_week'] >= 5).astype(np.int8)
hourly_full['is_holiday']  = hourly_full['hour_ts'].dt.date.apply(
    lambda x: 1 if x in holiday_dates else 0
).astype(np.int8)
hourly_full['is_vacation'] = hourly_full['hour_ts'].dt.date.apply(
    lambda x: 1 if x in vacation_dates else 0
).astype(np.int8)

# Zyklische Kodierung
hourly_full['hour_sin']  = np.sin(2 * np.pi * hourly_full['hour_of_day'] / 24)
hourly_full['hour_cos']  = np.cos(2 * np.pi * hourly_full['hour_of_day'] / 24)
hourly_full['dow_sin']   = np.sin(2 * np.pi * hourly_full['day_of_week'] / 7)
hourly_full['dow_cos']   = np.cos(2 * np.pi * hourly_full['day_of_week'] / 7)
hourly_full['month_sin'] = np.sin(2 * np.pi * hourly_full['month'] / 12)
hourly_full['month_cos'] = np.cos(2 * np.pi * hourly_full['month'] / 12)

print(f'Kalender-Features hinzugefügt.')

# %%
# ── Wetter-Features joinen ──
hourly_full = hourly_full.merge(weather_hourly, on='hour_ts', how='left')

# Interpolation für kurze Lücken (max 6h), dann forward/backward fill
for col in ['temperature', 'humidity', 'precipitation', 'wind_speed']:
    hourly_full[col] = (
        hourly_full
        .groupby('station_id')[col]
        .transform(lambda x: x.interpolate(method='linear', limit=6).ffill().bfill())
    )

weather_cov = hourly_full['temperature'].notna().mean() * 100
print(f'Wetter-Coverage nach Interpolation: {weather_cov:.1f}%')

# ── Fehlende Wetterwerte mit Median füllen (Fallback) ──
for col in ['temperature', 'humidity', 'precipitation', 'wind_speed']:
    median_val = hourly_full[col].median()
    hourly_full[col] = hourly_full[col].fillna(median_val)

# %%
# ── Lag-Features (pro Station) ──
hourly_full = hourly_full.sort_values(['station_id', 'hour_ts']).reset_index(drop=True)

for lag_name, lag_hours in [('lag_1h', 1), ('lag_24h', 24), ('lag_168h', 168)]:
    hourly_full[f'demand_{lag_name}'] = (
        hourly_full
        .groupby('station_id')['total_demand']
        .shift(lag_hours)
    )

# Erste 168 Stunden pro Station haben NaN-Lags → entfernen
before = len(hourly_full)
hourly_full = hourly_full.dropna(subset=['demand_lag_168h']).reset_index(drop=True)
print(f'Lag-Features: {before - len(hourly_full):,} Zeilen entfernt (Anlaufphase)')


Kalender-Features hinzugefügt.
Wetter-Coverage nach Interpolation: 100.0%
Lag-Features: 17,472 Zeilen entfernt (Anlaufphase)


In [ ]:
# ## 5 — Statistisches Labeling

# %%
# ── Z-Score pro Station × Wochentag × Stunde ──
stats = (
    hourly_full
    .groupby(['station_id', 'day_of_week', 'hour_of_day'])['total_demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'hist_mean', 'std': 'hist_std'})
)

hourly_full = hourly_full.merge(
    stats, on=['station_id', 'day_of_week', 'hour_of_day'], how='left'
)

# std clippen, damit Division durch ≈0 keine Explosion gibt
hourly_full['hist_std'] = hourly_full['hist_std'].clip(lower=0.1)
hourly_full['z_score']  = (
    (hourly_full['total_demand'] - hourly_full['hist_mean']) / hourly_full['hist_std']
)

# Labels
hourly_full['label'] = 'normal'
hourly_full.loc[hourly_full['z_score'].abs() > Z_TRAIN_THRESHOLD, 'label'] = 'grauzone'
hourly_full.loc[hourly_full['z_score'].abs() > Z_ANOMALY_THRESHOLD, 'label'] = 'anomal'

print('Label-Verteilung:')
print(hourly_full['label'].value_counts())
print(f'\nAnomalie-Rate: {(hourly_full["label"] == "anomal").mean():.4f}')
print(f'Grauzone-Rate: {(hourly_full["label"] == "grauzone").mean():.4f}')

# %%
# ── Plausibilitätscheck: Bekannte Events ──
print('\n── Plausibilitätscheck: Anomalie-Rate an bekannten Tagen ──')
for date_obj in sorted(list(holiday_dates))[:10]:  # Erste 10 Feiertage
    day_data = hourly_full[hourly_full['hour_ts'].dt.date == date_obj]
    if len(day_data) == 0:
        continue
    anom_rate = (day_data['label'] == 'anomal').mean()
    grau_rate = (day_data['label'] == 'grauzone').mean()
    name = holidays_bw[holidays_bw['start_date'].dt.date == date_obj]['name'].values
    name = name[0] if len(name) > 0 else '?'
    print(f'  {date_obj} ({name}): anomal={anom_rate:.1%}, grauzone={grau_rate:.1%}')


Label-Verteilung:
label
normal      2460114
grauzone      67864
anomal        47998
Name: count, dtype: int64

Anomalie-Rate: 0.0186
Grauzone-Rate: 0.0263

── Plausibilitätscheck: Anomalie-Rate an bekannten Tagen ──


In [ ]:
# ══════════════════════════════════════════════════════════════
# KONTEXTUELLE FEATURES — vor dem Split einfügen (nach Schritt 5, vor Schritt 6/7)
# ══════════════════════════════════════════════════════════════

# Die z_scores und hist_mean/hist_std haben wir schon aus Schritt 5.
# Jetzt: Residuen als Features statt Rohwerte

# Kontextuelles Residuum: wie weit weicht die Nachfrage vom erwarteten Muster ab?
hourly_full['demand_residual'] = hourly_full['total_demand'] - hourly_full['hist_mean']
hourly_full['demand_ratio']    = hourly_full['total_demand'] / hourly_full['hist_mean'].clip(lower=0.1)

# Auch für Lags: Residuen statt Rohwerte
for lag_col in ['demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h']:
    hourly_full[f'{lag_col}_residual'] = hourly_full[lag_col] - hourly_full['hist_mean']

# ── Neue Feature-Liste ──
FEATURE_COLS = [
    # Kontextualisierte Nachfrage (DAS ist der Kern-Fix)
    'z_score',               # Wie anomal ist dieser Wert im Kontext?
    'demand_residual',       # Absolute Abweichung vom erwarteten Wert
    'demand_ratio',          # Relative Abweichung (2.0 = doppelt so viel wie erwartet)
    'total_demand',          # Rohwert bleibt als Referenz
    'n_lends', 'n_returns',
    # Kontextualisierte Lags
    'demand_lag_1h_residual',
    'demand_lag_24h_residual',
    'demand_lag_168h_residual',
    # Kalender (zyklisch)
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    # Kalender (binär)
    'is_weekend', 'is_holiday', 'is_vacation',
    # Wetter
    'temperature', 'humidity', 'precipitation', 'wind_speed',
]

# Demand-Feature Indices für Scoring (die ersten 9 sind demand-bezogen)
DEMAND_FEATURE_INDICES = list(range(9))

N_FEATURES = len(FEATURE_COLS)
INPUT_DIM_FLAT = WINDOW_SIZE * N_FEATURES

print(f'Neue Feature-Anzahl: {N_FEATURES}')
print(f'Demand-Features (für Scoring): {[FEATURE_COLS[i] for i in DEMAND_FEATURE_INDICES]}')
print(f'Flat input dim: {INPUT_DIM_FLAT}')

# ── z_score clippen (damit Extremwerte den Scaler nicht dominieren) ──
hourly_full['z_score'] = hourly_full['z_score'].clip(-10, 10)
hourly_full['demand_ratio'] = hourly_full['demand_ratio'].clip(0, 10)
for col in [c for c in FEATURE_COLS if 'residual' in c]:
    hourly_full[col] = hourly_full[col].clip(
        hourly_full[col].quantile(0.001),
        hourly_full[col].quantile(0.999)
    )

print(f'\nz_score Verteilung (Training = normal only):')
train_mask = (hourly_full['hour_ts'] < train_end) & (hourly_full['label'] == 'normal')
print(f'  Normal:  mean={hourly_full.loc[train_mask, "z_score"].mean():.3f}, '
      f'std={hourly_full.loc[train_mask, "z_score"].std():.3f}')
anom_mask = hourly_full['label'] == 'anomal'
print(f'  Anomal:  mean={hourly_full.loc[anom_mask, "z_score"].mean():.3f}, '
      f'std={hourly_full.loc[anom_mask, "z_score"].std():.3f}')

Neue Feature-Anzahl: 22
Demand-Features (für Scoring): ['z_score', 'demand_residual', 'demand_ratio', 'total_demand', 'n_lends', 'n_returns', 'demand_lag_1h_residual', 'demand_lag_24h_residual', 'demand_lag_168h_residual']
Flat input dim: 528

z_score Verteilung (Training = normal only):
  Normal:  mean=-0.168, std=0.613
  Anomal:  mean=4.476, std=1.578


In [ ]:
# ══════════════════════════════════════════════════════════════
# FIX: Bessere Anomalie-Definition + Point-wise Scoring
# ══════════════════════════════════════════════════════════════

# ── FIX 1: Anomalien brauchen Mindest-Aktivität ──
# Eine Station mit hist_mean=0.02 kann keine sinnvolle Anomalie produzieren.
# Mindestens 1 erwartete Ausleihe/Rückgabe UND mindestens 3 absolute Events.
MIN_HIST_MEAN = 1.0    # Station muss im Schnitt ≥1 Event/h haben für diesen Slot
MIN_ABSOLUTE  = 3      # Mindestens 3 Events absolut

hourly_full['label'] = 'normal'

is_anomaly = (
    (hourly_full['z_score'].abs() > Z_ANOMALY_THRESHOLD) &  # Statistisch auffällig
    (hourly_full['hist_mean'] >= MIN_HIST_MEAN) &            # Station ist aktiv genug
    (hourly_full['total_demand'] >= MIN_ABSOLUTE)            # Absolute Mindestmenge
)
is_grauzone = (
    (hourly_full['z_score'].abs() > Z_TRAIN_THRESHOLD) &
    ~is_anomaly  # Alles zwischen 2σ und "echte Anomalie"
)

hourly_full.loc[is_grauzone, 'label'] = 'grauzone'
hourly_full.loc[is_anomaly, 'label']  = 'anomal'

print('═══ Neue Label-Verteilung ═══')
print(hourly_full['label'].value_counts())
print(f'\nAnomalie-Rate: {is_anomaly.mean():.4f}')

# Wie sehen die neuen Anomalien aus?
new_anom = hourly_full[hourly_full['label'] == 'anomal']
print(f'\nNeue Anomalien — hist_mean:')
print(new_anom['hist_mean'].describe())
print(f'\nNeue Anomalien — total_demand:')
print(new_anom['total_demand'].describe())
print(f'\nBeispiele:')
print(new_anom.nlargest(10, 'z_score')[
    ['station_id', 'hour_ts', 'total_demand', 'hist_mean', 'hist_std', 'z_score']
].to_string(index=False))

═══ Neue Label-Verteilung ═══
label
normal      2460114
grauzone     101799
anomal        14063
Name: count, dtype: int64

Anomalie-Rate: 0.0055

Neue Anomalien — hist_mean:
count    14063.000000
mean         2.551699
std          2.089741
min          1.000000
25%          1.387755
50%          1.952703
75%          2.870748
max         26.641892
Name: hist_mean, dtype: float64

Neue Anomalien — total_demand:
count    14063.000000
mean        12.948233
std          8.199590
min          5.000000
25%          8.000000
50%         11.000000
75%         15.000000
max        119.000000
Name: total_demand, dtype: float64

Beispiele:
station_id                   hour_ts  total_demand  hist_mean  hist_std   z_score
  27230961 2024-03-15 17:00:00+00:00            57   1.714286  5.076133 10.000000
  27230961 2024-03-15 20:00:00+00:00            58   1.224490  4.930748 10.000000
  27230961 2025-04-13 13:00:00+00:00            44   1.452703  4.241573 10.000000
  27230961 2025-04-27 17:00:00+00:0

In [ ]:
# ## 6 — Parquet speichern (vor dem Split)

# %%
# ── Network-Name hinzufügen (für spätere Multi-City-Kompatibilität) ──
hourly_full['network_name'] = NETWORK_NAME

# ── Speichern ──
hourly_full.to_parquet(OUTPUT_PATH, index=False)
print(f'\n✅ Gespeichert: {OUTPUT_PATH}')
print(f'   Shape: {hourly_full.shape}')
print(f'   Spalten: {list(hourly_full.columns)}')
print(f'   Zeitraum: {hourly_full["hour_ts"].min().date()} – {hourly_full["hour_ts"].max().date()}')
print(f'   Stationen: {hourly_full["station_id"].nunique()}')



✅ Gespeichert: /content/data/Second_AD_Mannheim_only.parquet
   Shape: (2575976, 36)
   Spalten: ['station_id', 'hour_ts', 'n_lends', 'n_returns', 'total_demand', 'station_name_id', 'location_id', 'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'is_vacation', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'temperature', 'humidity', 'precipitation', 'wind_speed', 'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h', 'hist_mean', 'hist_std', 'z_score', 'label', 'demand_residual', 'demand_ratio', 'demand_lag_1h_residual', 'demand_lag_24h_residual', 'demand_lag_168h_residual', 'network_name']
   Zeitraum: 2023-04-07 – 2026-02-02
   Stationen: 104


In [ ]:
# ## 7 — Train / Val / Test Split

# %%
# ── Feature-Spalten definieren ──
FEATURE_COLS = [
    # Nachfrage
    'total_demand', 'n_lends', 'n_returns',
    'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
    # Kalender (zyklisch)
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    # Kalender (binär)
    'is_weekend', 'is_holiday', 'is_vacation',
    # Wetter
    'temperature', 'humidity', 'precipitation', 'wind_speed',
]
N_FEATURES = len(FEATURE_COLS)
print(f'Features: {N_FEATURES} → {FEATURE_COLS}')

# %%
# ── Zeitbasierter Split ──
t_min = hourly_full['hour_ts'].min()
t_max = hourly_full['hour_ts'].max()
total_hours = (t_max - t_min).total_seconds() / 3600

train_end = t_min + pd.Timedelta(hours=int(total_hours * TRAIN_RATIO))
val_end   = t_min + pd.Timedelta(hours=int(total_hours * VAL_RATIO))

# Train: NUR normale Daten
df_train = hourly_full[(hourly_full['hour_ts'] < train_end) &
                        (hourly_full['label'] == 'normal')].copy()
# Val & Test: ALLE Daten (inkl. Anomalien)
df_val   = hourly_full[(hourly_full['hour_ts'] >= train_end) &
                        (hourly_full['hour_ts'] < val_end)].copy()
df_test  = hourly_full[hourly_full['hour_ts'] >= val_end].copy()

print(f'Train (nur normal): {len(df_train):>8,} | bis {train_end.date()}')
print(f'Val   (alle):       {len(df_val):>8,} | '
      f'Anomalien: {(df_val["label"]=="anomal").sum():,} '
      f'({(df_val["label"]=="anomal").mean():.2%})')
print(f'Test  (alle):       {len(df_test):>8,} | '
      f'Anomalien: {(df_test["label"]=="anomal").sum():,} '
      f'({(df_test["label"]=="anomal").mean():.2%})')

# %%
# ── Normalisierung (Scaler nur auf Train fitten!) ──
scaler = StandardScaler()
train_scaled = scaler.fit_transform(df_train[FEATURE_COLS].values)
val_scaled   = scaler.transform(df_val[FEATURE_COLS].values)
test_scaled  = scaler.transform(df_test[FEATURE_COLS].values)

print(f'Scaler fitted auf {len(train_scaled):,} Trainings-Samples')

# %%
# ── Sequenzen bilden (pro Station, keine Überlappung über Stationsgrenzen) ──
def create_station_sequences(df, scaled_data, window_size):
    """
    Erstellt Sequenzen der Länge window_size pro Station.
    Gibt (sequences, labels, metadata) zurück.
    """
    sequences, labels, meta = [], [], []
    station_ids = df['station_id'].values
    timestamps  = df['hour_ts'].values
    label_arr   = df['label'].values

    for station in df['station_id'].unique():
        mask = station_ids == station
        s_data   = scaled_data[mask]
        s_labels = label_arr[mask]
        s_times  = timestamps[mask]

        if len(s_data) < window_size:
            continue

        for i in range(len(s_data) - window_size + 1):
            sequences.append(s_data[i:i + window_size])
            labels.append(s_labels[i + window_size - 1])  # Label des letzten Zeitpunkts
            meta.append({
                'station_id': station,
                'timestamp': s_times[i + window_size - 1]
            })

    return np.array(sequences, dtype=np.float32), np.array(labels), meta

print(f'Sequenzen erstellen (Window={WINDOW_SIZE})...')
X_train, y_train, meta_train = create_station_sequences(df_train, train_scaled, WINDOW_SIZE)
X_val,   y_val,   meta_val   = create_station_sequences(df_val,   val_scaled,   WINDOW_SIZE)
X_test,  y_test,  meta_test  = create_station_sequences(df_test,  test_scaled,  WINDOW_SIZE)

print(f'X_train: {X_train.shape}  (nur normal)')
print(f'X_val:   {X_val.shape}    (Anomalien: {(y_val=="anomal").sum()})')
print(f'X_test:  {X_test.shape}   (Anomalien: {(y_test=="anomal").sum()})')

# Für Vanilla AE und VAE: flatten
INPUT_DIM_FLAT = WINDOW_SIZE * N_FEATURES
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
print(f'\nFlattened dim: {INPUT_DIM_FLAT}')

Features: 19 → ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_holiday', 'is_vacation', 'temperature', 'humidity', 'precipitation', 'wind_speed']
Train (nur normal): 1,655,156 | bis 2025-02-27
Val   (alle):        412,152 | Anomalien: 3,021 (0.73%)
Test  (alle):        438,048 | Anomalien: 2,026 (0.46%)
Scaler fitted auf 1,655,156 Trainings-Samples
Sequenzen erstellen (Window=24)...
X_train: (1652764, 24, 19)  (nur normal)
X_val:   (409760, 24, 19)    (Anomalien: 3016)
X_test:  (435656, 24, 19)   (Anomalien: 2012)

Flattened dim: 456


In [ ]:
# ## 8 — Modellarchitekturen

# %%
# ══════════════════════════════════════════════════════════════
# 8a — Vanilla Autoencoder (Baseline)
# ══════════════════════════════════════════════════════════════
class VanillaAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def get_latent(self, x):
        return self.encoder(x)


# ══════════════════════════════════════════════════════════════
# 8b — LSTM-Autoencoder (Hauptmodell)
# ══════════════════════════════════════════════════════════════
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=32, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim
        self.n_layers = n_layers

        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=latent_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.decoder = nn.LSTM(
            input_size=latent_dim,
            hidden_size=latent_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.output_layer = nn.Linear(latent_dim, n_features)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        batch_size, seq_len, _ = x.shape

        # Encode → nur letzter Hidden State
        _, (hidden, cell) = self.encoder(x)

        # Decode: Latent-Vektor als Eingabe für jeden Zeitschritt wiederholen
        latent = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)  # (batch, seq_len, latent_dim)
        decoder_out, _ = self.decoder(latent, (hidden, cell))

        # Output-Projektion
        reconstruction = self.output_layer(decoder_out)
        return reconstruction

    def get_latent(self, x):
        _, (hidden, _) = self.encoder(x)
        return hidden[-1]


# ══════════════════════════════════════════════════════════════
# 8c — Variational Autoencoder (VAE)
# ══════════════════════════════════════════════════════════════
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu

In [ ]:
# ## 9 — Training Framework

# %%
def train_ae(model, X_train, X_val, model_name='AE',
             epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE,
             is_vae=False, vae_beta=1.0):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    criterion = nn.MSELoss()
    scaler_amp = torch.amp.GradScaler('cuda')  # ← NEU

    train_tensor = torch.FloatTensor(X_train).to(device)
    train_loader = DataLoader(
        TensorDataset(train_tensor, train_tensor),
        batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=0, pin_memory=False  # Daten sind schon auf GPU
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)  # ← schneller als zero_grad()

            with torch.amp.autocast('cuda'):  # ← Mixed Precision
                if is_vae:
                    x_hat, mu, logvar = model(batch_x)
                    recon_loss = criterion(x_hat, batch_x)
                    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = recon_loss + vae_beta * kl_loss
                else:
                    x_hat = model(batch_x)
                    loss = criterion(x_hat, batch_x)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

# ── Validation (nur alle 3 Epochs — spart ~30% Zeit) ──
        # ── Validation (chunked, alle 3 Epochs) ──
        if (epoch + 1) % 3 == 0 or epoch == 0 or epoch == epochs - 1:
            model.eval()
            val_losses = []
            val_chunk_size = 4096
            with torch.no_grad(), torch.amp.autocast('cuda'):
                for vi in range(0, len(X_val), val_chunk_size):
                    v_chunk = torch.FloatTensor(X_val[vi:vi+val_chunk_size]).to(device)
                    if is_vae:
                        v_hat, v_mu, v_logvar = model(v_chunk)
                        v_recon = criterion(v_hat, v_chunk)
                        v_kl = -0.5 * torch.mean(1 + v_logvar - v_mu.pow(2) - v_logvar.exp())
                        val_losses.append((v_recon + vae_beta * v_kl).item())
                    else:
                        v_hat = model(v_chunk)
                        val_losses.append(criterion(v_hat, v_chunk).item())
                    del v_chunk, v_hat
            val_loss = np.mean(val_losses)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            val_loss = history['val_loss'][-1] if history['val_loss'] else avg_train_loss
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  [{model_name}] Epoch {epoch+1:>3}/{epochs} | '
                  f'Train: {avg_train_loss:.6f} | Val: {val_loss:.6f} | LR: {lr_now:.1e}')
    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)
    print(f'  [{model_name}] Best Val Loss: {best_val_loss:.6f}\n')
    return history


# ── Indices der Demand-Features im Feature-Vektor ──
DEMAND_FEATURE_INDICES = [
    FEATURE_COLS.index('total_demand'),
    FEATURE_COLS.index('n_lends'),
    FEATURE_COLS.index('n_returns'),
    FEATURE_COLS.index('demand_lag_1h'),
    FEATURE_COLS.index('demand_lag_24h'),
    FEATURE_COLS.index('demand_lag_168h'),
]
print(f'Demand-Feature Indices: {DEMAND_FEATURE_INDICES}')
print(f'Demand-Features: {[FEATURE_COLS[i] for i in DEMAND_FEATURE_INDICES]}')

def compute_anomaly_scores(model, X, is_vae=False, chunk_size=2048,
                           demand_indices=DEMAND_FEATURE_INDICES):
    """Rekonstruktionsfehler NUR auf Demand-Features."""
    model.eval()
    model = model.to(device)
    all_scores = []

    for i in range(0, len(X), chunk_size):
        chunk = torch.FloatTensor(X[i:i+chunk_size]).to(device)
        with torch.no_grad(), torch.amp.autocast('cuda'):
            if is_vae:
                x_hat, _, _ = model(chunk)
            else:
                x_hat = model(chunk)

            # Nur Demand-Features für den Score
            if chunk.dim() == 3:
                # LSTM: (batch, seq_len, features) → nur letzte features-Dim filtern
                diff = (chunk[:, :, demand_indices] - x_hat[:, :, demand_indices]) ** 2
            else:
                # Flat: (batch, seq_len*features) → reshape, filtern, flatten
                batch_size = chunk.shape[0]
                c_3d = chunk.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                h_3d = x_hat.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                diff = (c_3d[:, :, demand_indices] - h_3d[:, :, demand_indices]) ** 2

            scores = diff.mean(dim=tuple(range(1, diff.dim())))

        all_scores.append(scores.cpu())
        del chunk, x_hat, scores, diff
        torch.cuda.empty_cache()

    return torch.cat(all_scores).numpy()


def find_best_threshold(scores, labels):
    """Schwellenwert der den F1-Score auf Val maximiert."""
    binary = (labels == 'anomal').astype(int)
    if binary.sum() == 0:
        return np.percentile(scores, 95), 0.0
    prec, rec, thresholds = precision_recall_curve(binary, scores)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1[:-1])
    return thresholds[best_idx], f1[best_idx]


def evaluate_model(model, X, y, threshold, model_name='AE', is_vae=False):
    """Vollständige Evaluation mit allen Metriken."""
    scores = compute_anomaly_scores(model, X, is_vae=is_vae)
    binary = (y == 'anomal').astype(int)
    preds  = (scores > threshold).astype(int)

    # AUC-PR (Hauptmetrik)
    prec, rec, _ = precision_recall_curve(binary, scores)
    auc_pr = auc(rec, prec)

    # F1
    f1 = f1_score(binary, preds, zero_division=0)

    # AUC-ROC
    try:
        auc_roc = roc_auc_score(binary, scores)
    except ValueError:
        auc_roc = 0.0

    print(f'── {model_name} auf Testdaten ──')
    print(f'  AUC-PR:  {auc_pr:.4f}  (Hauptmetrik)')
    print(f'  F1:      {f1:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  Threshold: {threshold:.6f}')
    print(classification_report(binary, preds, target_names=['Normal', 'Anomal'],
                                 zero_division=0))

    return {
        'model': model_name, 'auc_pr': auc_pr, 'f1': f1, 'auc_roc': auc_roc,
        'threshold': threshold, 'scores': scores, 'preds': preds
    }

Demand-Feature Indices: [0, 1, 2, 3, 4, 5]
Demand-Features: ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h']


In [ ]:
# ── Nach Sequenzbildung einfügen (vor dem Training) ──
# Prüfe wie groß die Daten tatsächlich sind
print(f'X_train: {X_train.shape} → {X_train.nbytes / 1e9:.2f} GB')
print(f'X_val:   {X_val.shape}   → {X_val.nbytes / 1e9:.2f} GB')
print(f'X_test:  {X_test.shape}  → {X_test.nbytes / 1e9:.2f} GB')

X_train: (1652764, 24, 19) → 3.01 GB
X_val:   (409760, 24, 19)   → 0.75 GB
X_test:  (435656, 24, 19)  → 0.79 GB


In [ ]:
# ══════════════════════════════════════════════════════════════
# MODELL 1: Vanilla Autoencoder (Baseline)
# ══════════════════════════════════════════════════════════════
torch.cuda.empty_cache()

vanilla_ae = VanillaAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vanilla_ae.parameters()):,}')

hist_vanilla = train_ae(vanilla_ae, X_train_flat, X_val_flat,
                        model_name='VanillaAE', epochs=30, batch_size=8192)

scores_val_v = compute_anomaly_scores(vanilla_ae, X_val_flat)
thresh_v, f1_v = find_best_threshold(scores_val_v, y_val)
print(f'Val-Threshold: {thresh_v:.6f} (F1={f1_v:.4f})')
results_vanilla = evaluate_model(vanilla_ae, X_test_flat, y_test, thresh_v, 'VanillaAE')

# Vom GPU räumen
vanilla_ae = vanilla_ae.cpu()
torch.save(vanilla_ae.state_dict(), '/content/data/vanilla_ae_mannheim.pth')
torch.cuda.empty_cache()
print('✅ Vanilla AE gespeichert & GPU freigegeben')

Parameter: 308,456
  [VanillaAE] Epoch   1/30 | Train: 0.392083 | Val: 0.331860 | LR: 1.0e-03
  [VanillaAE] Epoch   5/30 | Train: 0.188469 | Val: 0.255963 | LR: 1.0e-03
  [VanillaAE] Epoch  10/30 | Train: 0.164072 | Val: 0.191965 | LR: 1.0e-03
  [VanillaAE] Epoch  15/30 | Train: 0.153229 | Val: 0.169360 | LR: 1.0e-03
  [VanillaAE] Epoch  20/30 | Train: 0.147666 | Val: 0.163268 | LR: 1.0e-03
  [VanillaAE] Epoch  25/30 | Train: 0.144739 | Val: 0.156955 | LR: 1.0e-03
  [VanillaAE] Epoch  30/30 | Train: 0.142533 | Val: 0.152145 | LR: 1.0e-03
  [VanillaAE] Best Val Loss: 0.152145

Val-Threshold: 0.944262 (F1=0.0540)
── VanillaAE auf Testdaten ──
  AUC-PR:  0.0235  (Hauptmetrik)
  F1:      0.0479
  AUC-ROC: 0.8607
  Threshold: 0.944262
              precision    recall  f1-score   support

      Normal       1.00      0.94      0.97    433644
      Anomal       0.03      0.32      0.05      2012

    accuracy                           0.94    435656
   macro avg       0.51      0.63      0.5

In [ ]:
# ══════════════════════════════════════════════════════════════
# MODELL 2: LSTM-Autoencoder (Hauptmodell)
# ══════════════════════════════════════════════════════════════
torch.cuda.empty_cache()
print(f'GPU frei: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')

lstm_ae = LSTMAutoencoder(n_features=N_FEATURES, latent_dim=LATENT_DIM, n_layers=2)
print(f'Parameter: {sum(p.numel() for p in lstm_ae.parameters()):,}')

# LSTM braucht 3D-Tensoren → viel mehr VRAM
# → Kleinerer Batch, Val auf CPU in Chunks berechnen
hist_lstm = train_ae(lstm_ae, X_train, X_val,
                     model_name='LSTM-AE', epochs=30, batch_size=2048)

scores_val_l = compute_anomaly_scores(lstm_ae, X_val)
thresh_l, f1_l = find_best_threshold(scores_val_l, y_val)
print(f'Val-Threshold: {thresh_l:.6f} (F1={f1_l:.4f})')
results_lstm = evaluate_model(lstm_ae, X_test, y_test, thresh_l, 'LSTM-AE')

lstm_ae = lstm_ae.cpu()
torch.save(lstm_ae.state_dict(), '/content/data/lstm_ae_mannheim.pth')
torch.cuda.empty_cache()
print('✅ LSTM-AE gespeichert & GPU freigegeben')

GPU frei: 20.1 GB / 23.7 GB
Parameter: 32,755


KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════
# MODELL 3: Variational Autoencoder (VAE)
# ══════════════════════════════════════════════════════════════
torch.cuda.empty_cache()
print(f'GPU frei: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')

vae = VAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vae.parameters()):,}')

hist_vae = train_ae(vae, X_train_flat, X_val_flat,
                    model_name='VAE', epochs=30, batch_size=8192,
                    is_vae=True, vae_beta=0.5)

scores_val_vae = compute_anomaly_scores(vae, X_val_flat, is_vae=True)
thresh_vae, f1_vae = find_best_threshold(scores_val_vae, y_val)
print(f'Val-Threshold: {thresh_vae:.6f} (F1={f1_vae:.4f})')
results_vae = evaluate_model(vae, X_test_flat, y_test, thresh_vae, 'VAE', is_vae=True)

vae = vae.cpu()
torch.save(vae.state_dict(), '/content/data/vae_mannheim.pth')
torch.cuda.empty_cache()
print('✅ VAE gespeichert & GPU freigegeben')

In [ ]:
# ── Alle Modelle neu scoren mit Demand-only-Score ──
torch.cuda.empty_cache()

# Vanilla AE
vanilla_ae = vanilla_ae.to(device)
scores_val_v = compute_anomaly_scores(vanilla_ae, X_val_flat)
thresh_v, f1_v = find_best_threshold(scores_val_v, y_val)
results_vanilla = evaluate_model(vanilla_ae, X_test_flat, y_test, thresh_v, 'VanillaAE')
vanilla_ae = vanilla_ae.cpu()
torch.cuda.empty_cache()


# LSTM-AE
lstm_ae = lstm_ae.to(device)
scores_val_l = compute_anomaly_scores(lstm_ae, X_val, chunk_size=1024)
thresh_l, f1_l = find_best_threshold(scores_val_l, y_val)
results_lstm = evaluate_model(lstm_ae, X_test, y_test, thresh_l, 'LSTM-AE')
lstm_ae = lstm_ae.cpu()
torch.cuda.empty_cache()

# VAE
vae = vae.to(device)
scores_val_vae = compute_anomaly_scores(vae, X_val_flat, is_vae=True)
thresh_vae, f1_vae = find_best_threshold(scores_val_vae, y_val)
results_vae = evaluate_model(vae, X_test_flat, y_test, thresh_vae, 'VAE', is_vae=True)
vae = vae.cpu()
torch.cuda.empty_cache()


# Vergleich
print('\n' + '='*50)
print('MODELLVERGLEICH (Demand-Only Scoring)')
print('='*50)
for r in [results_vanilla, results_lstm, results_vae]:
    print(f'{r["model"]:<12} AUC-PR={r["auc_pr"]:.4f}  F1={r["f1"]:.4f}  AUC-ROC={r["auc_roc"]:.4f}')

In [ ]:
def compute_anomaly_scores(model, X, is_vae=False, chunk_size=4096):
    """Rekonstruktionsfehler pro Sample — chunked für große Daten."""
    model.eval()
    model = model.to(device)
    all_scores = []

    for i in range(0, len(X), chunk_size):
        chunk = torch.FloatTensor(X[i:i+chunk_size]).to(device)
        with torch.no_grad(), torch.amp.autocast('cuda'):
            if is_vae:
                x_hat, _, _ = model(chunk)
            else:
                x_hat = model(chunk)
            scores = ((chunk - x_hat) ** 2).mean(
                dim=tuple(range(1, chunk.dim()))
            )
        all_scores.append(scores.cpu())
        del chunk, x_hat, scores
        torch.cuda.empty_cache()

    return torch.cat(all_scores).numpy()

In [ ]:
# ══════════════════════════════════════════════════════════════
# DIAGNOSE: Warum funktioniert die AD nicht?
# ══════════════════════════════════════════════════════════════

# 1. Was sind unsere "Anomalien" eigentlich?
anomalies = hourly_full[hourly_full['label'] == 'anomal']
normals   = hourly_full[hourly_full['label'] == 'normal']

print('═══ DIAGNOSE 1: Anomalie-Charakteristik ═══')
print(f'\nAnzahl Anomalien: {len(anomalies):,} ({len(anomalies)/len(hourly_full):.2%})')
print(f'\nTotal Demand bei Anomalien:')
print(anomalies['total_demand'].describe())
print(f'\nTotal Demand bei Normaldaten:')
print(normals['total_demand'].describe())
print(f'\nhist_mean bei Anomalien:')
print(anomalies['hist_mean'].describe())
print(f'\nhist_std bei Anomalien:')
print(anomalies['hist_std'].describe())

# 2. Wie viele Anomalien sind an Low-Activity-Stationen?
print('\n═══ DIAGNOSE 2: Low-Count-Problem ═══')
print(f'Anomalien mit hist_mean < 1:  {(anomalies["hist_mean"] < 1).sum():,} '
      f'({(anomalies["hist_mean"] < 1).mean():.1%})')
print(f'Anomalien mit hist_mean < 0.5: {(anomalies["hist_mean"] < 0.5).sum():,} '
      f'({(anomalies["hist_mean"] < 0.5).mean():.1%})')
print(f'Anomalien mit total_demand <= 3: {(anomalies["total_demand"] <= 3).sum():,} '
      f'({(anomalies["total_demand"] <= 3).mean():.1%})')

# 3. Beispiel: Was sieht eine typische "Anomalie" aus?
print('\n═══ DIAGNOSE 3: Typische Anomalien ═══')
sample = anomalies.nlargest(10, 'z_score')[
    ['station_id', 'hour_ts', 'total_demand', 'hist_mean', 'hist_std', 'z_score']
]
print(sample.to_string(index=False))

print('\n═══ DIAGNOSE 3b: Typische LOW-COUNT Anomalien ═══')
low_count_anom = anomalies[anomalies['hist_mean'] < 0.5].sample(
    min(10, len(anomalies[anomalies['hist_mean'] < 0.5])), random_state=42
)[['station_id', 'hour_ts', 'total_demand', 'hist_mean', 'hist_std', 'z_score']]
print(low_count_anom.to_string(index=False))

# 4. Score-Verteilung: Sind Anomalien überhaupt separierbar?
print('\n═══ DIAGNOSE 4: Score-Separierbarkeit ═══')
# Vanilla AE Scores auf Testdaten
vanilla_ae = vanilla_ae.to(device)
test_scores = compute_anomaly_scores(vanilla_ae, X_test_flat)
vanilla_ae = vanilla_ae.cpu()
torch.cuda.empty_cache()

test_binary = (y_test == 'anomal')
print(f'Score Normal:  mean={test_scores[~test_binary].mean():.4f}, '
      f'median={np.median(test_scores[~test_binary]):.4f}, '
      f'p95={np.percentile(test_scores[~test_binary], 95):.4f}')
print(f'Score Anomal:  mean={test_scores[test_binary].mean():.4f}, '
      f'median={np.median(test_scores[test_binary]):.4f}, '
      f'p95={np.percentile(test_scores[test_binary], 95):.4f}')
print(f'Overlap: Die Verteilungen überlappen {"stark" if test_scores[test_binary].mean() < np.percentile(test_scores[~test_binary], 75) else "moderat"}')

# 5. Sequenz-Problem: Label ist letzter Zeitpunkt, Score ist Durchschnitt über 24h
print('\n═══ DIAGNOSE 5: Sequenz-Verwässerung ═══')
print(f'Window size: {WINDOW_SIZE}h')
print(f'Label basiert auf: letztem Zeitpunkt der Sequenz')
print(f'Score basiert auf: Durchschnitt über alle {WINDOW_SIZE} Zeitpunkte')
print(f'→ Wenn nur 1 von {WINDOW_SIZE} Stunden anomal ist, wird das Signal 1/{WINDOW_SIZE} so stark')

═══ DIAGNOSE 1: Anomalie-Charakteristik ═══

Anzahl Anomalien: 47,998 (1.86%)

Total Demand bei Anomalien:
count    47998.000000
mean         5.958040
std          6.694429
min          1.000000
25%          2.000000
50%          4.000000
75%          8.000000
max        119.000000
Name: total_demand, dtype: float64

Total Demand bei Normaldaten:
count    2.460114e+06
mean     1.159110e+00
std      2.594533e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      6.000000e+01
Name: total_demand, dtype: float64

hist_mean bei Anomalien:
count    47998.000000
mean         0.934013
std          1.552521
min          0.006757
25%          0.094595
50%          0.317568
75%          1.216216
max         26.641892
Name: hist_mean, dtype: float64

hist_std bei Anomalien:
count    47998.000000
mean         1.194388
std          1.280179
min          0.100000
25%          0.352544
50%          0.756051
75%          1.617022
max         19.199335
Name: